<a href="https://colab.research.google.com/github/vadhan123/NLP/blob/main/2266_NLP_LAB_ASSIGNMENT_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.3 MB/s eta 0:00:00


NUMERICAL VECTOR REPRESENTATION WITH WORD2VEC

In [ ]:
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from gensim.models import Word2Vec

In [ ]:
texts=[
    "movie is good it is value for price","movie is bad and boring"
    ]

labels = [1,0]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

print(word_index)

{'is': 1, 'movie': 2, 'good': 3, 'it': 4, 'value': 5, 'for': 6, 'price': 7, 'bad': 8, 'and': 9, 'boring': 10}


In [ ]:
print(sequences)

[[2, 1, 3, 4, 1, 5, 6, 7], [2, 1, 8, 9, 10]]


In [ ]:
max_length=6
X = pad_sequences(sequences, maxlen=max_length)
y = np.array(labels)



In [ ]:
X

array([[ 3,  4,  1,  5,  6,  7],
       [ 0,  2,  1,  8,  9, 10]], dtype=int32)

In [ ]:
sentences = [text.split() for text in texts]
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [ ]:
vocab_size = len(word_index) + 1
embedding_dim = 50
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]

In [ ]:
embedding_matrix[1]

array([-1.06877461e-03,  4.72547486e-04,  1.02054179e-02,  1.80089008e-02,
       -1.86171979e-02, -1.42446533e-02,  1.29111288e-02,  1.79373957e-02,
       -1.00387093e-02, -7.53464736e-03,  1.47689842e-02, -3.07443761e-03,
       -9.06358380e-03,  1.31108295e-02, -9.73028969e-03, -3.62294074e-03,
        5.76698175e-03,  1.99467130e-03, -1.65809579e-02, -1.88875422e-02,
        1.46253724e-02,  1.01497835e-02,  1.35188391e-02,  1.52974587e-03,
        1.27008455e-02, -6.79855375e-03, -1.88420305e-03,  1.15466118e-02,
       -1.50428331e-02, -7.87793938e-03, -1.50271673e-02, -1.87325443e-03,
        1.90703273e-02, -1.46409273e-02, -4.67035593e-03, -3.88620375e-03,
        1.61554497e-02, -1.18689919e-02,  9.38261073e-05, -9.51052550e-03,
       -1.92111060e-02,  1.00184558e-02, -1.75076164e-02, -8.79075937e-03,
       -6.04289853e-05, -5.91265096e-04, -1.53137315e-02,  1.92337502e-02,
        9.97366197e-03,  1.84616502e-02])

In [ ]:
input_layer = Input(shape=(max_length,))


In [ ]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_length,
    trainable=False
)(input_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

In [ ]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [ ]:
merged = Concatenate()([pool1, pool2, pool3])


In [ ]:
output = Dense(1, activation='sigmoid')(merged)

In [ ]:
model = Model(inputs=input_layer, outputs=output)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │        550 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        301 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 61,151 (238.87 KB)

 Trainable params: 60,601 (236.72 KB)

 Non-trainable params: 550 (2.15 KB)

In [ ]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=2
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6931
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6823
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 0.6719
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6622
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6530
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 0.6442
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - loss: 0.6357
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6277
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6197
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 0.6116


In [ ]:
test_text = ["this movie is amazing"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
[[0.49601492]]
